In [ ]:
!pip install --quiet sacrebleu rouge-score nltk

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Comparing text automatically

- BLEU, ROUGE and METEOR score the overlap between an answer and a reference
- They measure shared words, which says little about whether the answer is correct
- Where overlap says nothing useful, one model can judge another's output

Three evaluators run over the same pair of texts.

### Exercise BLEU, ROUGE, METEOR
Run the code below and compare the metric results for comparing the same 2 texts using different metrics.
Note the differences in the results and the need to use the same metric when comparing texts.

In [ ]:
%pip install sacrebleu rouge-score nltk sacrebleu rouge-score

In [ ]:
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

prediction = "Your refund is approved and arrives in 3 days"
reference = "The refund has been approved, expect it in 3 days"

# BLEU
bleu = BLEU(effective_order=True)  # better behavior on short texts
bleu_score = bleu.corpus_score([prediction], [[reference]]).score / 100.0  # normalize 0-1

# ROUGE
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge_scores = scorer.score(reference, prediction)
rougeL_f = rouge_scores["rougeL"].fmeasure

# METEOR
meteor = meteor_score([reference.split()], prediction.split())

print(f"BLEU: {bleu_score:.4f}")
print(f"ROUGE-L (F1): {rougeL_f:.4f}")
print(f"METEOR: {meteor:.4f}")

### Exercise
A/B tests
Replace the texts in the example below with our own and check which one the model selects.

In [ ]:
from langchain_classic.evaluation import load_evaluator
import json
from dotenv import load_dotenv

load_dotenv()

evaluator = load_evaluator("labeled_pairwise_string", llm=make_llm())

result = evaluator.evaluate_string_pairs(
    input="When will my refund arrive?",
    prediction="Your refund is approved and arrives in 3 days",
    prediction_b="I don't know",
    reference="The refund is approved and lands within three days"
)

print(json.dumps(result, indent=4))

### Exercise
Embedding Distance Evaluator
Run the code below and answer the question:
When is the embedding distance greatest - when the texts are similar, or when the texts differ significantly from each other?

In [ ]:
from langchain_classic.evaluation import load_evaluator
from langchain_openai import OpenAIEmbeddings

embeddings = make_embeddings()
evaluator = load_evaluator("embedding_distance", embeddings=embeddings)

result1 = evaluator.evaluate_strings(
    prediction="The refund has been approved, expect it in 3 days",
    reference="The refund has been approved, expect it in 3 days"
)

result2 = evaluator.evaluate_strings(
    prediction="The refund has been approved, expect it in 3 days",
    reference="Poland's capital is named Warsaw"
)

result3 = evaluator.evaluate_strings(
    prediction="The refund has been approved, expect it in 3 days",
    reference="The capital of Burkina Faso is named Ouagadougou"
)

print(round(result1["score"], 4))
print(round(result2["score"], 4))
print(round(result3["score"], 4))

### Try a right answer, worded differently

- Rewrite the prediction so it means the same thing in different words
- The overlap scores drop while the answer stays correct. That gap is why
  these metrics are a smoke test and not a verdict